# LLM Inference Optimization: KV Cache, PagedAttention, Continuous Batching, Speculative Decoding & Quantization

Serving Large Language Models cost-effectively at low latency requires mastering the hardware and algorithmic limits:
1. **Memory Bandwidth vs Compute (The Roofline Model)**: Why autoregressive decoding is memory-bandwidth bound at small batch sizes.
2. **KV Cache Footprint & GQA**: Sizing memory consumption across MHA, Grouped Query Attention (GQA), and Multi-Query Attention (MQA).
3. **PagedAttention & Continuous Batching (vLLM style)**: Virtual memory page tables for non-contiguous KV cache allocation.
4. **Speculative Decoding**: Accelerating token generation using a small draft model verified in parallel by the target model.
5. **Post-Training Quantization (PTQ)**: Symmetric INT8 and blockwise INT4 quantization mechanics (AWQ and GPTQ concepts).

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('06-generative-ai/inference-optimization/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import inference_engine
print('inference_engine loaded')

## 1. Memory Bandwidth Roofline & Autoregressive Decoding Latency
During token generation (decode phase), batch size 1 requires loading every single model parameter from HBM to SRAM for every individual token.

In [ ]:
from inference_engine import memory_bandwidth_throughput_roofline

# Comparison: 7B vs 70B parameter models on an A100 GPU (1,000 GB/s HBM bandwidth)
bw_gpu = 1000.0  # GB/s

tok_7b = memory_bandwidth_throughput_roofline(7 * 10**9, bw_gpu, precision_bytes=2)
tok_70b = memory_bandwidth_throughput_roofline(70 * 10**9, bw_gpu, precision_bytes=2)

print(f"Theoretical peak decode throughput (batch=1) on A100 (1000 GB/s):")
print(f" - 7B Model  (14 GB weights): {tok_7b:.1f} tokens/second (~{1000/tok_7b:.1f} ms per token)")
print(f" - 70B Model (140 GB weights): {tok_70b:.1f} tokens/second (~{1000/tok_70b:.1f} ms per token)")

## 2. KV Cache Sizing: MHA vs GQA vs MQA
How Grouped Query Attention (GQA) dramatically reduces memory consumption per concurrent user.

In [ ]:
from inference_engine import kv_cache_size_bytes

batch = 16
seq_len = 4096
layers = 32
heads = 32
d_head = 128

# 1. Multi-Head Attention (32 KV heads)
bytes_mha = kv_cache_size_bytes(batch, seq_len, layers, num_kv_heads=32, head_dim=d_head)

# 2. Grouped Query Attention (8 KV heads, e.g. LLaMA 3 8B)
bytes_gqa = kv_cache_size_bytes(batch, seq_len, layers, num_kv_heads=8, head_dim=d_head)

# 3. Multi-Query Attention (1 KV head)
bytes_mqa = kv_cache_size_bytes(batch, seq_len, layers, num_kv_heads=1, head_dim=d_head)

print(f"KV Cache Size for {batch} concurrent streams at {seq_len} context length:")
print(f" - Multi-Head Attention (MHA) : {bytes_mha / (1024**3):.2f} GiB")
print(f" - Grouped-Query Attention(GQA): {bytes_gqa / (1024**3):.2f} GiB ({bytes_mha/bytes_gqa:.1f}x smaller)")
print(f" - Multi-Query Attention (MQA) : {bytes_mqa / (1024**3):.2f} GiB ({bytes_mha/bytes_mqa:.1f}x smaller)")

## 3. PagedAttention Virtual Block Allocator (vLLM Pattern)
Mapping contiguous logical token positions to non-contiguous physical memory blocks to eliminate fragmentation.

In [ ]:
from inference_engine import PagedBlockAllocator

# Create allocator with 8 physical blocks of size 16 tokens each
allocator = PagedBlockAllocator(num_blocks=8, block_size=16)

# Request A arrives: Prompt has 28 tokens -> allocated 2 blocks
req_a_blocks = allocator.allocate("req_A", num_tokens=28)
print(f"Request A allocated physical blocks: {req_a_blocks} (utilization={allocator.memory_utilization(8)*100:.1f}%)")

# Request B arrives: Prompt has 14 tokens -> allocated 1 block
req_b_blocks = allocator.allocate("req_B", num_tokens=14)
print(f"Request B allocated physical blocks: {req_b_blocks} (utilization={allocator.memory_utilization(8)*100:.1f}%)")

# Request A generates 4 tokens (28 -> 32 tokens) -> hits block boundary
allocator.append_token("req_A", current_token_count=32)
print(f"Request A after crossing boundary: {allocator.block_table['req_A']}")

# Request A finishes and frees blocks
allocator.free("req_A")
print(f"After freeing Request A: {len(allocator.free_blocks)} blocks available for new requests.")

## 4. Speculative Decoding Simulation
A fast draft model generates $K$ candidate tokens; the large target model verifies them in parallel in one forward pass.

In [ ]:
import torch
from inference_engine import speculative_decoding_step

# Fast draft model and accurate target model
vocab_size = 50

def draft_model(ctx):
    logits = torch.randn(vocab_size)
    logits[7] = 8.0  # draft tends to predict token 7
    return logits

def target_model(ctx):
    logits = torch.randn(vocab_size)
    logits[7] = 12.0  # target also confirms token 7
    return logits

prefix = [1, 10]
accepted, count = speculative_decoding_step(draft_model, target_model, prefix, gamma=4)

print(f"Speculative step proposed 4 tokens.")
print(f"Tokens accepted + bonus: {count} tokens emitted in a single target model pass!")
print(f"Accepted sequence: {accepted}")

## 5. Weight Quantization: Symmetric INT8 & Grouped INT4
Compressing model parameters to fit in GPU memory and double effective memory bandwidth.

In [ ]:
import torch
from inference_engine import (
    quantize_int8_symmetric, dequantize_int8_symmetric,
    quantize_int4_blockwise, dequantize_int4_blockwise
)

torch.manual_seed(42)
weights_fp32 = torch.randn(256, 256)

# 1. Symmetric INT8 Quantization
q8, scale8 = quantize_int8_symmetric(weights_fp32)
w_rec_8 = dequantize_int8_symmetric(q8, scale8)
mse_8 = torch.mean((weights_fp32 - w_rec_8) ** 2).item()
print(f"INT8 Quantization: Compression=4x  | Reconstruction MSE={mse_8:.6f}")

# 2. Blockwise Grouped INT4 Quantization (group_size=32)
q4, scales4 = quantize_int4_blockwise(weights_fp32, group_size=32)
w_rec_4 = dequantize_int4_blockwise(q4, scales4, weights_fp32.numel(), weights_fp32.shape)
mse_4 = torch.mean((weights_fp32 - w_rec_4) ** 2).item()
print(f"INT4 Quantization: Compression=8x  | Reconstruction MSE={mse_4:.6f}")